# SVM Breast Cancer Image Classification

In [1]:
# Chạy local: mở notebook từ thư mục Buoi_4 hoặc thư mục con.
from pathlib import Path  # Dùng đường dẫn đa nền tảng
ROOT = next((p for p in [Path.cwd(), *Path.cwd().parents]  # Tìm thư mục dự án từ vị trí hiện tại
             if (p / 'Data' / 'dicom_info.csv').is_file()), None)  # Chọn thư mục có dữ liệu
if ROOT is None:  # Kiểm tra đã tìm thấy dự án
    raise FileNotFoundError('Không tìm thấy Data/dicom_info.csv. Mở Jupyter từ thư mục Buoi_4.')  # Báo lỗi khi thiếu thư mục dữ liệu
DATA_DIR = ROOT / 'Data'  # Đặt thư mục dữ liệu
MODEL_DIR = ROOT / 'models'  # Đặt thư mục lưu mô hình
MODEL_DIR.mkdir(exist_ok=True)  # Tạo thư mục nếu chưa có
# True: kiểm tra nhanh trên 80 dòng train và 40 dòng test. False: dùng toàn bộ dữ liệu.
QUICK_RUN = False  # Bật hoặc tắt chạy nhanh
# Bật khi muốn tìm tham số; chậm hơn huấn luyện SVM mặc định.
RUN_GRID_SEARCH = False  # Bật hoặc tắt tìm tham số
print('Thư mục dự án:', ROOT)  # In thư mục đang sử dụng


Thư mục dự án: /Volumes/Workspace /Cybersoft/ML07/Buoi4_Code


In [3]:
!python -m pip install -r requirements.txt

1. Import các thư viện code

In [6]:
import matplotlib.pyplot as plt # Dùng hàm vẽ biểu đồ
import seaborn as sns # Dùng biểu đồ thống kê 
import numpy as np
import pandas as pd 
from sklearn.model_selection import train_test_split
from sklearn.svm import SVC
from sklearn.metrics import classification_report, confusion_matrix
from sklearn.preprocessing import StandardScaler # Dùng bộ chuẩn hóa đặc trưng
import matplotlib.image as mpimg
from skimage.feature import hog # Trích xuất đặc trưng HOG
from skimage.transform import resize # Thay đổi kích thước ảnh
import os 

In [7]:
# Đọc file meta.csv -> mô tả các series ảnh của Data
df_meta = pd.read_csv(DATA_DIR / 'meta.csv')
df_meta.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 6775 entries, 0 to 6774
Data columns (total 9 columns):
 #   Column             Non-Null Count  Dtype 
---  ------             --------------  ----- 
 0   SeriesInstanceUID  6775 non-null   object
 1   StudyInstanceUID   6775 non-null   object
 2   Modality           6775 non-null   object
 3   SeriesDescription  6775 non-null   object
 4   BodyPartExamined   6775 non-null   object
 5   SeriesNumber       6775 non-null   int64 
 6   Collection         6775 non-null   object
 7   Visibility         6775 non-null   int64 
 8   ImageCount         6775 non-null   int64 
dtypes: int64(3), object(6)
memory usage: 476.5+ KB


In [9]:
 # Đọc file dicom_info.csv
df_dicom = pd.read_csv(DATA_DIR / 'dicom_info.csv')
df_meta.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 6775 entries, 0 to 6774
Data columns (total 9 columns):
 #   Column             Non-Null Count  Dtype 
---  ------             --------------  ----- 
 0   SeriesInstanceUID  6775 non-null   object
 1   StudyInstanceUID   6775 non-null   object
 2   Modality           6775 non-null   object
 3   SeriesDescription  6775 non-null   object
 4   BodyPartExamined   6775 non-null   object
 5   SeriesNumber       6775 non-null   int64 
 6   Collection         6775 non-null   object
 7   Visibility         6775 non-null   int64 
 8   ImageCount         6775 non-null   int64 
dtypes: int64(3), object(6)
memory usage: 476.5+ KB


In [10]:
# Liệt kê các loại ảnh 
df_dicom.SeriesDescription.unique()

array(['cropped images', 'full mammogram images', nan, 'ROI mask images'],
      dtype=object)

In [ ]:
# Liệt kê các ảnh Crop
cropped_images = df_dicom[df_dicom.SeriesDescription=='cropped images'].image_path
cropped_images.head(5)

0     CBIS-DDSM/jpeg/1.3.6.1.4.1.9590.100.1.2.129308...
3     CBIS-DDSM/jpeg/1.3.6.1.4.1.9590.100.1.2.381187...
6     CBIS-DDSM/jpeg/1.3.6.1.4.1.9590.100.1.2.153339...
7     CBIS-DDSM/jpeg/1.3.6.1.4.1.9590.100.1.2.178994...
10    CBIS-DDSM/jpeg/1.3.6.1.4.1.9590.100.1.2.411833...
Name: image_path, dtype: object

In [12]:
# Lọc các đường dẫn ảnh ROI
roi_img = df_dicom[df_dicom.SeriesDescription=='ROI mask images'].image_path
roi_img.head(5)

5     CBIS-DDSM/jpeg/1.3.6.1.4.1.9590.100.1.2.153339...
8     CBIS-DDSM/jpeg/1.3.6.1.4.1.9590.100.1.2.178994...
9     CBIS-DDSM/jpeg/1.3.6.1.4.1.9590.100.1.2.411833...
14    CBIS-DDSM/jpeg/1.3.6.1.4.1.9590.100.1.2.236373...
20    CBIS-DDSM/jpeg/1.3.6.1.4.1.9590.100.1.2.357008...
Name: image_path, dtype: object

In [17]:
#full mammogram images
full_mammo = df_dicom[df_dicom.SeriesDescription=='full mammogram images'].image_path  # Lọc đường dẫn ảnh toàn cảnh
full_mammo.head(5)  # Xem 5 đường dẫn đầu


1     CBIS-DDSM/jpeg/1.3.6.1.4.1.9590.100.1.2.248386...
2     CBIS-DDSM/jpeg/1.3.6.1.4.1.9590.100.1.2.267213...
11    CBIS-DDSM/jpeg/1.3.6.1.4.1.9590.100.1.2.210396...
12    CBIS-DDSM/jpeg/1.3.6.1.4.1.9590.100.1.2.749566...
15    CBIS-DDSM/jpeg/1.3.6.1.4.1.9590.100.1.2.987658...
Name: image_path, dtype: object

In [15]:
# Đường dẫn thư mục ảnh JPEG
imdir = str(DATA_DIR / 'jpeg')
print(imdir)

/Volumes/Workspace /Cybersoft/ML07/Buoi4_Code/Data/jpeg


In [18]:
# Chỉ thay tiền tố của dataset bằng thư mục ảnh local.
def local_image_path(path):  # Định nghĩa hàm đổi đường dẫn
    parts = Path(path).parts  # Tách các phần đường dẫn
    return str(DATA_DIR / 'jpeg' / parts[-2] / parts[-1])  # Ghép UID và tên ảnh vào thư mục local
cropped_images = cropped_images.map(local_image_path)  # Đổi đường dẫn ảnh crop
full_mammo = full_mammo.map(local_image_path)  # Đổi đường dẫn ảnh toàn cảnh
roi_img = roi_img.map(local_image_path)  # Đổi đường dẫn ảnh mặt nạ
print('Ảnh crop mẫu:', cropped_images.iloc[0])  # In đường dẫn ảnh mẫu


Ảnh crop mẫu: /Volumes/Workspace /Cybersoft/ML07/Buoi4_Code/Data/jpeg/1.3.6.1.4.1.9590.100.1.2.129308726812851964007517874181459556304/1-172.jpg


In [19]:
# Tạo bảng tra cứu UID -> đường dẫn; crop và ảnh toàn cảnh có UID riêng 
full_mammo_dict = dict()
cropped_images_dict = dict()
roi_img_dict = dict()

for dicom in full_mammo:
    key = dicom.split("/")[-2]
    full_mammo_dict[key] = dicom

for dicom in cropped_images:
    key = dicom.split("/")[-2]
    cropped_images_dict[key] = dicom

for dicom in roi_img:
    key = dicom.split("/")[-2]
    roi_img_dict[key] = dicom

print('Full mammo dict key example: ')
print(next(iter(full_mammo_dict.items())))
print('\nROI mask images dict key example: ')
print(next(iter(roi_img_dict.items())))

Full mammo dict key example: 
('1.3.6.1.4.1.9590.100.1.2.248386742010678582309005372213277814849', '/Volumes/Workspace /Cybersoft/ML07/Buoi4_Code/Data/jpeg/1.3.6.1.4.1.9590.100.1.2.248386742010678582309005372213277814849/1-249.jpg')

ROI mask images dict key example: 
('1.3.6.1.4.1.9590.100.1.2.153339052913121382622526066491844156138', '/Volumes/Workspace /Cybersoft/ML07/Buoi4_Code/Data/jpeg/1.3.6.1.4.1.9590.100.1.2.153339052913121382622526066491844156138/2-270.jpg')


In [20]:
mass_train = pd.read_csv(DATA_DIR / 'mass_case_description_train_set.csv')  # Đọc dữ liệu huấn luyện
mass_test = pd.read_csv(DATA_DIR / 'mass_case_description_test_set.csv')  # Đọc dữ liệu kiểm tra
mass_train.head()  # Xem các dòng train đầu


,patient_id,breast_density,left or right breast,image view,abnormality id,abnormality type,mass shape,mass margins,assessment,pathology,subtlety,image file path,cropped image file path,ROI mask file path
0,P_00001,3,LEFT,CC,1,mass,IRREGULAR-ARCHITECTURAL_DISTORTION,SPICULATED,4,MALIGNANT,4,Mass-Training_P_00001_LEFT_CC/1.3.6.1.4.1.9590...,Mass-Training_P_00001_LEFT_CC_1/1.3.6.1.4.1.95...,Mass-Training_P_00001_LEFT_CC_1/1.3.6.1.4.1.95...
1,P_00001,3,LEFT,MLO,1,mass,IRREGULAR-ARCHITECTURAL_DISTORTION,SPICULATED,4,MALIGNANT,4,Mass-Training_P_00001_LEFT_MLO/1.3.6.1.4.1.959...,Mass-Training_P_00001_LEFT_MLO_1/1.3.6.1.4.1.9...,Mass-Training_P_00001_LEFT_MLO_1/1.3.6.1.4.1.9...
2,P_00004,3,LEFT,CC,1,mass,ARCHITECTURAL_DISTORTION,ILL_DEFINED,4,BENIGN,3,Mass-Training_P_00004_LEFT_CC/1.3.6.1.4.1.9590...,Mass-Training_P_00004_LEFT_CC_1/1.3.6.1.4.1.95...,Mass-Training_P_00004_LEFT_CC_1/1.3.6.1.4.1.95...
3,P_00004,3,LEFT,MLO,1,mass,ARCHITECTURAL_DISTORTION,ILL_DEFINED,4,BENIGN,3,Mass-Training_P_00004_LEFT_MLO/1.3.6.1.4.1.959...,Mass-Training_P_00004_LEFT_MLO_1/1.3.6.1.4.1.9...,Mass-Training_P_00004_LEFT_MLO_1/1.3.6.1.4.1.9...
4,P_00004,3,RIGHT,MLO,1,mass,OVAL,CIRCUMSCRIBED,4,BENIGN,5,Mass-Training_P_00004_RIGHT_MLO/1.3.6.1.4.1.95...,Mass-Training_P_00004_RIGHT_MLO_1/1.3.6.1.4.1....,Mass-Training_P_00004_RIGHT_MLO_1/1.3.6.1.4.1....


In [21]:
# CSV chứa đường dẫn DICOM; dùng UID thư mục để tìm đúng ảnh JPEG.
# Truy cập theo tên cột để không phụ thuộc thứ tự cột trong CSV.
def fix_image_path(data):  # Định nghĩa hàm sửa đường dẫn
    for column, lookup in [('image file path', full_mammo_dict),  # Chọn cột ảnh toàn cảnh và bảng tra
                           ('cropped image file path', cropped_images_dict)]:  # Chọn cột ảnh crop và bảng tra
        paths = []  # Tạo danh sách đường dẫn mới
        for value in data[column]:  # Duyệt từng đường dẫn cũ
            uid = str(value).strip().split('/')[-2]  # Lấy UID và bỏ khoảng trắng
            if uid not in lookup:  # Kiểm tra UID trong bảng tra
                raise ValueError(f'Không tìm thấy UID {uid} cho {column}')  # Báo lỗi nếu thiếu UID
            path = lookup[uid]  # Lấy đường dẫn JPEG
            if not Path(path).is_file():  # Kiểm tra file ảnh tồn tại
                raise FileNotFoundError(f'Thiếu ảnh: {path}')  # Báo lỗi nếu thiếu ảnh
            paths.append(path)  # Thêm đường dẫn hợp lệ
        data[column] = paths  # Cập nhật cột đường dẫn
fix_image_path(mass_train)  # Sửa đường dẫn tập train
fix_image_path(mass_test)  # Sửa đường dẫn tập test


In [22]:
# Liệt kê các nhãn bệnh lý 
mass_train.pathology.unique()

array(['MALIGNANT', 'BENIGN', 'BENIGN_WITHOUT_CALLBACK'], dtype=object)

In [23]:
# rename columns
mass_train = mass_train.rename(columns={'left or right breast': 'left_or_right_breast',  # Đổi tên cột bên ngực
                                           'image view': 'image_view',  # Đổi tên cột góc chụp
                                           'abnormality id': 'abnormality_id',  # Đổi tên cột mã bất thường
                                           'abnormality type': 'abnormality_type',  # Đổi tên cột loại bất thường
                                           'mass shape': 'mass_shape',  # Đổi tên cột hình dạng khối
                                           'mass margins': 'mass_margins',  # Đổi tên cột bờ khối
                                           'image file path': 'image_file_path',  # Đổi tên cột ảnh toàn cảnh
                                           'cropped image file path': 'cropped_image_file_path',  # Đổi tên cột ảnh crop
                                           'ROI mask file path': 'ROI_mask_file_path'})  # Đổi tên cột mặt nạ ROI

mass_train.head(5)  # Xem 5 dòng sau đổi tên


,patient_id,breast_density,left_or_right_breast,image_view,abnormality_id,abnormality_type,mass_shape,mass_margins,assessment,pathology,subtlety,image_file_path,cropped_image_file_path,ROI_mask_file_path
0,P_00001,3,LEFT,CC,1,mass,IRREGULAR-ARCHITECTURAL_DISTORTION,SPICULATED,4,MALIGNANT,4,/Volumes/Workspace /Cybersoft/ML07/Buoi4_Code/...,/Volumes/Workspace /Cybersoft/ML07/Buoi4_Code/...,Mass-Training_P_00001_LEFT_CC_1/1.3.6.1.4.1.95...
1,P_00001,3,LEFT,MLO,1,mass,IRREGULAR-ARCHITECTURAL_DISTORTION,SPICULATED,4,MALIGNANT,4,/Volumes/Workspace /Cybersoft/ML07/Buoi4_Code/...,/Volumes/Workspace /Cybersoft/ML07/Buoi4_Code/...,Mass-Training_P_00001_LEFT_MLO_1/1.3.6.1.4.1.9...
2,P_00004,3,LEFT,CC,1,mass,ARCHITECTURAL_DISTORTION,ILL_DEFINED,4,BENIGN,3,/Volumes/Workspace /Cybersoft/ML07/Buoi4_Code/...,/Volumes/Workspace /Cybersoft/ML07/Buoi4_Code/...,Mass-Training_P_00004_LEFT_CC_1/1.3.6.1.4.1.95...
3,P_00004,3,LEFT,MLO,1,mass,ARCHITECTURAL_DISTORTION,ILL_DEFINED,4,BENIGN,3,/Volumes/Workspace /Cybersoft/ML07/Buoi4_Code/...,/Volumes/Workspace /Cybersoft/ML07/Buoi4_Code/...,Mass-Training_P_00004_LEFT_MLO_1/1.3.6.1.4.1.9...
4,P_00004,3,RIGHT,MLO,1,mass,OVAL,CIRCUMSCRIBED,4,BENIGN,5,/Volumes/Workspace /Cybersoft/ML07/Buoi4_Code/...,/Volumes/Workspace /Cybersoft/ML07/Buoi4_Code/...,Mass-Training_P_00004_RIGHT_MLO_1/1.3.6.1.4.1....


In [25]:
mass_train['mass_shape'] = mass_train['mass_shape'].bfill()
mass_train['mass_margins'] = mass_train['mass_margins'].bfill()

mass_train.isnull().sum()

patient_id                 0
breast_density             0
left_or_right_breast       0
image_view                 0
abnormality_id             0
abnormality_type           0
mass_shape                 0
mass_margins               0
assessment                 0
pathology                  0
subtlety                   0
image_file_path            0
cropped_image_file_path    0
ROI_mask_file_path         0
dtype: int64

In [27]:
mass_test = mass_test.rename(columns={'left or right breast': 'left_or_right_breast',  # Đổi tên cột bên ngực
                                           'image view': 'image_view',  # Đổi tên cột góc chụp
                                           'abnormality id': 'abnormality_id',  # Đổi tên cột mã bất thường
                                           'abnormality type': 'abnormality_type',  # Đổi tên cột loại bất thường
                                           'mass shape': 'mass_shape',  # Đổi tên cột hình dạng khối
                                           'mass margins': 'mass_margins',  # Đổi tên cột bờ khối
                                           'image file path': 'image_file_path',  # Đổi tên cột ảnh toàn cảnh
                                           'cropped image file path': 'cropped_image_file_path',  # Đổi tên cột ảnh crop
                                           'ROI mask file path': 'ROI_mask_file_path'})  # Đổi tên cột mặt nạ ROI

mass_test.head(5)


,patient_id,breast_density,left_or_right_breast,image_view,abnormality_id,abnormality_type,mass_shape,mass_margins,assessment,pathology,subtlety,image_file_path,cropped_image_file_path,ROI_mask_file_path
0,P_00016,4,LEFT,CC,1,mass,IRREGULAR,SPICULATED,5,MALIGNANT,5,/Volumes/Workspace /Cybersoft/ML07/Buoi4_Code/...,/Volumes/Workspace /Cybersoft/ML07/Buoi4_Code/...,Mass-Test_P_00016_LEFT_CC_1/1.3.6.1.4.1.9590.1...
1,P_00016,4,LEFT,MLO,1,mass,IRREGULAR,SPICULATED,5,MALIGNANT,5,/Volumes/Workspace /Cybersoft/ML07/Buoi4_Code/...,/Volumes/Workspace /Cybersoft/ML07/Buoi4_Code/...,Mass-Test_P_00016_LEFT_MLO_1/1.3.6.1.4.1.9590....
2,P_00017,2,LEFT,CC,1,mass,ROUND,CIRCUMSCRIBED,4,MALIGNANT,4,/Volumes/Workspace /Cybersoft/ML07/Buoi4_Code/...,/Volumes/Workspace /Cybersoft/ML07/Buoi4_Code/...,Mass-Test_P_00017_LEFT_CC_1/1.3.6.1.4.1.9590.1...
3,P_00017,2,LEFT,MLO,1,mass,ROUND,ILL_DEFINED,4,MALIGNANT,4,/Volumes/Workspace /Cybersoft/ML07/Buoi4_Code/...,/Volumes/Workspace /Cybersoft/ML07/Buoi4_Code/...,Mass-Test_P_00017_LEFT_MLO_1/1.3.6.1.4.1.9590....
4,P_00032,3,RIGHT,CC,1,mass,ROUND,OBSCURED,0,BENIGN,2,/Volumes/Workspace /Cybersoft/ML07/Buoi4_Code/...,/Volumes/Workspace /Cybersoft/ML07/Buoi4_Code/...,Mass-Test_P_00032_RIGHT_CC_1/1.3.6.1.4.1.9590....


In [28]:
# fill in missing values using the backwards fill method
mass_test['mass_margins'] = mass_test['mass_margins'].bfill()  # Điền bờ khối thiếu từ dòng sau

#check null values
mass_test.isnull().sum()  # Đếm giá trị thiếu theo cột


patient_id                 0
breast_density             0
left_or_right_breast       0
image_view                 0
abnormality_id             0
abnormality_type           0
mass_shape                 0
mass_margins               0
assessment                 0
pathology                  0
subtlety                   0
image_file_path            0
cropped_image_file_path    0
ROI_mask_file_path         0
dtype: int64